In [ ]:
import os
import subprocess
import time

import flatdict as fd
import requests
import yaml

print(os.getcwd())

In [ ]:
url = "https://ndownloader.figshare.com/files/42742570"
output = "test.pos"


def download_requests(url: str, output_file_path: str) -> bool:
    r = requests.get(url, stream=True, allow_redirects=True)
    if r.status_code == 200:
        print(f"{r.url}, {r.status_code}, {r.headers}")
        with open(output_file_path, "wb") as fp:
            for chunk in r.iter_content(1024 * 1024):
                fp.write(chunk)
        if int(os.path.getsize(output)) == int(r.headers["Content-Length"]):
            return True
        else:
            return False
    else:
        return False


status = download_requests(url, output)
print(status)

In [ ]:
with open("data/datasets.yaml", encoding="utf-8") as fp:
    datasets = fd.FlatDict(yaml.safe_load(fp) or {}, delimiter="/")
for key, value in datasets.items():
    if key.endswith(r"\@origin"):
        case = key.rsplit("/", 1)[0]
        if os.path.isfile(f"data/{value}"):  # local file, never compressed by default
            print(f"local file, not compressed >>>> data/{value}")
        elif value.startswith("https://"):  # file to download
            if value.count(":") == 2:  # compressed
                archive, file_name = value.rsplit(":", 1)
                print(f"remote file, compressed >>>> {archive}, {file_name}")
            else:
                print(f"remote file, not compressed >>>> {value}")
        else:
            continue

***

In [ ]:
os.path.getsize(output)

In [ ]:
def download_requests(url: str, output_name: str, retries: int = 10) -> bool:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/139.0 Safari/537.36"
        ),
        "Accept": "*/*",
    }

    with requests.Session() as session:
        session.headers.update(headers)

        for attempt in range(retries):
            try:
                print(f"Attempt {attempt + 1}/{retries}")

                response = session.get(
                    url,
                    stream=True,
                    allow_redirects=True,
                    timeout=(30, 300),
                )

                print(f"HTTP {response.status_code} {response.url}")

                if response.status_code == 202:
                    response.close()
                    time.sleep(2)
                    continue

                response.raise_for_status()

                tmp = output_name + ".tmp"
                size = 0

                with open(tmp, "wb") as f:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                            size += len(chunk)

                response.close()

                if size == 0:
                    raise RuntimeError("Downloaded 0 bytes")

                os.replace(tmp, output_name)

                print(f"Downloaded {size:,} bytes")
                return True

            except (requests.RequestException, OSError, RuntimeError) as e:
                print(f"Failed: {e}")
                time.sleep(2)

    return False

In [ ]:
def download_wget(url: str, output_name: str) -> bool:
    result = subprocess.run(
        [
            "wget",
            '--user-agent="Mozilla/5.0"',
            "--content-disposition",
            "--tries=10",
            "--waitretry=5",
            "--retry-on-http-error=202",
            "-O",
            output_name,
            url,
        ],
        check=False,
    )

    return result.returncode == 0